# 🌧️ Daily Rainfall Forecasting — End-to-End Time Series Project
**Project:** PRAICP-1004 · Rainfall Time Series (Changi Climate Station, Singapore)
**Role:** AI Engineer
**Version:** daily forecasting (rebuilt from the original *monthly-total* notebook)

## 1. Business Case
The Meteorological Service Singapore / NEA record rainfall every day at the **Changi Climate Station**.
A *daily* forecast is what operations actually use: **drainage and flood preparedness, outdoor-event and construction scheduling, airport/aviation operations and water-resource management** all decide day by day, not month by month.

## 2. Project Goal
Predict **daily rainfall** at Changi for the coming days, and select the best model objectively. For every future day the model returns:
1. **P(rain)** — the probability that the day gets ≥ 0.2 mm (the official "rain day" definition),
2. **Expected rainfall (mm)**, and
3. a **95 % upper bound (mm)** for planning against heavy-rain days.

## 3. Why daily is a different problem from monthly (and what changed in this notebook)
| | Monthly (old notebook) | **Daily (this notebook)** |
|---|---|---|
| Observations | 462 | ~14 000 |
| Target distribution | right-skewed, never 0 | **zero-inflated**: a large share of days are dry, wet days are extremely skewed |
| Seasonal cycle | 12 points / year | 365 points / year → Fourier terms or day-of-year climatology |
| Memory | weak (lag-1 ACF ≈ 0.2) | weather spells (wet/dry runs) → **short memory of a few days** |
| Right loss / metrics | RMSE, MAE, MASE | RMSE/MAE **plus probabilistic scores** (Brier score, AUC, interval coverage) — MAPE is impossible with zero days |
| Models | SARIMA, Holt-Winters (log), ... | log transform no longer works (zeros) → **two-part "hurdle" models**: *does it rain?* × *how much?* |
| Horizon | 12 months | **1, 3 and 7 days ahead** (skill decays quickly with lead time) |
| Validation | 5 folds × 12 months | 5 folds × 365 days, evaluated at each lead time |

## 4. Data
Daily rainfall total (mm) for Changi, one row per day. The notebook accepts either
- **one CSV** with a date column and a daily rainfall column, or
- **a folder of the Meteorological Service Singapore "Daily Records" monthly files** (`DAILYDATA_S24_YYYYMM.csv`; columns *Station, Year, Month, Day, Daily Rainfall Total (mm), …*). Missing values written as `-` are handled.

Set `DATA_PATH` in section 2.1 (or drop the file/folder next to the notebook as `daily_rainfall.csv` / `data/`).
> ⚠️ **If no daily data is found, the notebook generates SYNTHETIC daily rainfall so that it still runs end-to-end, and prints a warning.** Never report results from synthetic data — point `DATA_PATH` at the real files first.

## 5. Roadmap
| Step | What we do |
|---|---|
| 1 | Load, audit (continuity, missing days, logical checks) |
| 2 | **EDA** – zero-inflation, wet-day amounts, seasonality, trend, wet/dry spells (Markov), ACF/PACF, stationarity, predictability ceiling |
| 3 | Chronological **train / test split** + **rolling-origin cross-validation** (no shuffling, no leakage), at lead times of 1, 3, 7 days |
| 4 | Models, each fixing a weakness of the previous: baselines → climatology → harmonic hurdle → hurdle with weather-memory (GLM) → ML (gradient boosting, random forest) |
| 5 | Compare (CV + hold-out test), pick the best, check **calibration** of the rain probabilities |
| 6 | Re-fit on all data, forecast the **next 7 days** with probabilities and an upper bound |
| 7 | Save the model + metadata, reload it and forecast (deployment check) |

> Each code cell is followed by a **💡 Insight** cell explaining how to read the output and what decision it drives. Cells also print an `INSIGHT:` line computed from the actual data, so the numbers always match your run.

## 1. Setup & Imports

In [1]:
import warnings, json, time, itertools, pathlib, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import roc_auc_score

import joblib

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
SEED = 42
np.random.seed(SEED)

# ---- project-wide settings --------------------------------------------------------------------
WET_THR   = 0.2           # a "rain day" has >= 0.2 mm (official NEA/MSS definition)
TEST_DAYS = 730           # hold-out test = last 2 years
CV_H, CV_STEP, N_FOLDS = 365, 365, 5     # rolling-origin CV: 5 folds, each validates 365 days
HORIZONS  = (1, 3, 7)     # lead times (days ahead) that are evaluated
H_MAIN    = 1             # headline lead time for plots
print("pandas", pd.__version__, "| statsmodels", sm.__version__)

pandas 2.2.3 | statsmodels 0.14.5


### 💡 Insight
- A fixed seed and a **chronological** evaluation design make every comparison reproducible and fair.
- `WET_THR = 0.2 mm` follows the official rain-day definition, so *"does it rain?"* is a well-defined yes/no event we can score with probabilities.
- Daily forecasts are evaluated at several **lead times** (1, 3, 7 days ahead): rain is far easier to predict tomorrow than next week, and the notebook shows exactly how fast the skill disappears.

## 2. Load & Audit the Data

### 2.1 Load the daily rainfall series

In [ ]:
DATA_PATH = None     # e.g. "daily_rainfall.csv"  or  "Data/daily"  (a CSV file, or a folder of CSV files)


def load_daily_rainfall(path):
    # Load daily rainfall (mm) from a CSV file or a folder of CSV files (MSS 'DAILYDATA_*' format or generic date,rainfall).
    path = pathlib.Path(path)
    files = [path] if path.is_file() else sorted(f for f in path.rglob("C:\\Users\\Girish M\\Downloads\\PRAICP-1004-RainfallTS\\Data\\rainfall-monthly-highest-daily-total.csv") if "monthly" not in f.name.lower())
    if not files:
        raise FileNotFoundError(f"No daily CSV files found in {path}")
    frames = []
    for f in files:
        t = pd.read_csv(f, encoding="latin-1", na_values=["-", "--", "NA", "N/A", "", " "], low_memory=False)
        t.columns = [str(c).strip() for c in t.columns]
        frames.append(t)
    t = pd.concat(frames, ignore_index=True)
    low = {c.lower(): c for c in t.columns}

    # keep one station (Changi) if several are present
    if "station" in low:
        st = t[low["station"]].astype(str)
        keep = st.str.contains("changi", case=False)
        t = t[keep] if keep.any() else t[st == st.mode()[0]]

    # --- rainfall column
    rain_col = next((c for c in t.columns if "daily rainfall total" in c.lower()), None)
    if rain_col is None:
        rain_col = next((c for c in t.columns if "rain" in c.lower() and "highest" not in c.lower() and "30" not in c and "60" not in c and "120" not in c), None)
    if rain_col is None:
        raise ValueError(f"Could not find a daily-rainfall column in {list(t.columns)}")

    # --- date
    date_col = next((low[k] for k in ("date", "datetime", "day_date") if k in low), None)
    if date_col is not None:
        dates = pd.to_datetime(t[date_col], errors="coerce")
    elif {"year", "month", "day"} <= set(low):
        dates = pd.to_datetime(dict(year=t[low["year"]], month=t[low["month"]], day=t[low["day"]]), errors="coerce")
    else:
        raise ValueError(f"Could not find a date (or Year/Month/Day) column in {list(t.columns)}")

    s = pd.Series(pd.to_numeric(t[rain_col], errors="coerce").values, index=dates, name="rain_mm")
    s = s[s.index.notna()]
    s = s[~s.index.duplicated(keep="first")].sort_index()
    print(f"Loaded {len(files)} file(s); rainfall column = '{rain_col}'")
    return s


def synthetic_daily_rainfall(seed=42):
    # Stand-in data (Markov wet/dry chain + seasonal gamma amounts) so the notebook runs end-to-end. NOT real data.
    rng = np.random.default_rng(seed)
    idx = pd.date_range("1982-01-01", "2020-06-30", freq="D")
    doy = idx.dayofyear.values
    season = np.cos(2 * np.pi * (doy - 355) / 365.25)           # peaks in December
    p_base, mu_wet = 0.42 + 0.14 * season, 11 + 5 * season
    y, prev = np.zeros(len(idx)), False
    for i in range(len(idx)):
        p = p_base[i] + (0.22 * (1 - p_base[i]) if prev else -0.12 * p_base[i])
        prev = rng.random() < p
        y[i] = max(WET_THR, round(rng.gamma(0.7, mu_wet[i] / 0.7), 1)) if prev else 0.0
    return pd.Series(y, index=idx, name="rain_mm")


candidates = [DATA_PATH] if DATA_PATH else [pathlib.Path.cwd() / n for n in ("daily_rainfall.csv", "Data", "data")]
found = next((pathlib.Path(p) for p in candidates if p and pathlib.Path(p).exists()), None)
USING_SYNTHETIC = False
if found is not None:
    try:
        rain = load_daily_rainfall(found)
    except (FileNotFoundError, ValueError) as e:
        print("Could not use", found, "->", e); found = None
if found is None:
    rain, USING_SYNTHETIC = synthetic_daily_rainfall(SEED), True
    print("!" * 100)
    print("WARNING: no daily rainfall data found -> using SYNTHETIC data. Set DATA_PATH to your real daily files.")
    print("         All numbers below are for demonstration only and say nothing about real Changi rainfall.")
    print("!" * 100)


# explicit daily frequency; any missing day becomes NaN
y_raw = rain.asfreq("D")
MISSING = y_raw.isna()
n_expected = (y_raw.index.max() - y_raw.index.min()).days + 1
longest_gap = (MISSING.groupby((~MISSING).cumsum()).sum()).max()
print(f"\nPeriod: {y_raw.index.min().date()} -> {y_raw.index.max().date()} | days: {len(y_raw)} (expected {n_expected})")
print(f"Missing days: {int(MISSING.sum())} ({MISSING.mean():.2%}) | longest gap: {int(longest_gap)} days")
display(y_raw.head(3).to_frame()); display(y_raw.tail(3).to_frame())

NotImplementedError: Non-relative patterns are unsupported

### 💡 Insight
- **What to check:** the number of rows equals the number of expected days (a gap-free daily calendar) and the share of missing days is small. A few percent missing is acceptable; long gaps (weeks) deserve investigation.
- **Decision:** missing days are *not* dropped (that would shift the 'yesterday' used by the memory features). They are filled with the **calendar-month average of the training period only** (next cell) and **excluded from every score**, so a filled value can never flatter or hurt a model.
- If the banner above says *SYNTHETIC*, nothing here is real data.

### 2.2 Handle missing days, logical checks and basic statistics

In [ ]:
# Fill missing days with the calendar-month mean of the TRAIN part only (no information from the test years)
n_train_days = len(y_raw) - TEST_DAYS
month_mean = y_raw.iloc[:n_train_days].groupby(y_raw.index[:n_train_days].month).mean()
y = y_raw.fillna(pd.Series(y_raw.index.month, index=y_raw.index).map(month_mean)).rename("rain_mm")
print(f"Filled {int(MISSING.sum())} missing day(s) | these days are excluded from all evaluation metrics.\n")

obs = y_raw.dropna()
checks = {"negative values": int((obs < 0).sum()),
          "days >= 250 mm (inspect: real storm or typo?)": int((obs >= 250).sum()),
          "values with resolution finer than 0.1 mm": int((((obs * 10) - (obs * 10).round()).abs() > 1e-6).sum())}
for k, v in checks.items():
    print(f"{k:50s}: {v}")

wet = obs[obs >= WET_THR]
top5 = obs.sort_values(ascending=False).head(int(0.05 * len(obs))).sum() / obs.sum()
summary = pd.Series({
    "days (observed)": len(obs),
    "dry days (< 0.2 mm), %": 100 * (obs < WET_THR).mean(),
    "rain days (>= 0.2 mm), %": 100 * (obs >= WET_THR).mean(),
    "mean daily rainfall (mm)": obs.mean(),
    "mean on rain days (mm)": wet.mean(),
    "median on rain days (mm)": wet.median(),
    "95th pct on rain days (mm)": wet.quantile(.95),
    "maximum (mm)": obs.max(),
    "skewness: all days": obs.skew(),
    "skewness: rain days only": wet.skew(),
    "share of total rain from the wettest 5% of days": 100 * top5,
})
display(summary.round(2).to_frame("value"))
print(f"INSIGHT: {100*(obs < WET_THR).mean():.0f}% of days are dry and the wettest 5% of days deliver {100*top5:.0f}% of all rain -> a zero-inflated, heavy-tailed target.")

### 💡 Insight
- **Zero-inflation:** a large fraction of daily values are exactly dry. A single regression on the raw values (or on `log(1+x)`) cannot represent *'it does not rain'* and *'how much when it does'* at the same time — this is why the monthly notebook's log-SARIMA approach does not transfer to daily data. We will model the two parts separately (**hurdle model**).
- **Heavy tail:** rain days are strongly right-skewed and the wettest few percent of days deliver a large share of the total. Those storm days dominate RMSE and matter most for flood planning, so they are **kept** (they are real events, not errors) — but the metrics must not rely on RMSE alone (we also score probabilities).
- The consistency checks replace the old *'rain days vs days-in-month'* checks (those needed the 3 monthly files). With daily data the only logical checks left are negatives and impossible values.

## 3. Exploratory Data Analysis (EDA)

### 3.1 The series over time

In [ ]:
monthly = y_raw.resample("MS").sum(min_count=20)
wetfreq365 = (y_raw >= WET_THR).where(y_raw.notna()).rolling(365, min_periods=300).mean()

fig, ax = plt.subplots(3, 1, figsize=(15, 10))
last2 = y.iloc[-730:]
ax[0].bar(last2.index, last2.values, width=1.0, color="steelblue"); ax[0].set_ylabel("mm"); ax[0].set_title("Daily rainfall — last 2 years")
ax[1].plot(monthly.index, monthly.values, lw=1, color="teal", label="monthly total (sum of daily)")
ax[1].plot(monthly.rolling(12).mean(), color="black", lw=2, label="12-month rolling mean"); ax[1].set_ylabel("mm"); ax[1].legend(); ax[1].set_title("Monthly totals aggregated from the daily data")
ax[2].plot(wetfreq365.index, 100 * wetfreq365.values, color="darkorange"); ax[2].set_ylabel("% of days"); ax[2].set_title("Share of rain days (365-day rolling)")
plt.tight_layout(); plt.show()

### 💡 Insight
- The daily series is **spiky**: long stretches of near-zero values interrupted by isolated heavy days. This is the raw material the forecast has to deal with — much noisier than the monthly view in panel 2.
- Aggregating the daily data to monthly totals (panel 2) lets you check the new dataset against the old monthly notebook: same yearly cycle, no obvious long-term drift in the 12-month mean.
- Panel 3 shows whether the *frequency* of rain days drifts over the years — a slow change there would call for a trend term in the occurrence model.

### 3.2 Distribution: the zero spike and the wet-day amounts

In [ ]:
w = obs[obs >= WET_THR]
a_hat, _, scale_hat = stats.gamma.fit(w, floc=0)
ks = stats.kstest(w, "gamma", args=(a_hat, 0, scale_hat))

fig, ax = plt.subplots(2, 3, figsize=(17, 9))
sns.histplot(obs.clip(upper=100), bins=60, ax=ax[0, 0], color="steelblue"); ax[0, 0].set_title("All days (clipped at 100 mm) — note the spike at 0")
ax[0, 1].hist(w, bins=np.logspace(np.log10(WET_THR), np.log10(w.max()), 40), color="darkorange"); ax[0, 1].set_xscale("log"); ax[0, 1].set_title("Rain-day amounts (log x-axis)")
sns.boxplot(x=w, ax=ax[0, 2], color="lightsteelblue"); ax[0, 2].set_xscale("log"); ax[0, 2].set_title("Rain-day amounts: boxplot (log x)")
stats.probplot(w, dist=stats.gamma, sparams=(a_hat, 0, scale_hat), plot=ax[1, 0]); ax[1, 0].set_title(f"Q-Q vs fitted Gamma (shape={a_hat:.2f})")
stats.probplot(np.log(w), dist="norm", plot=ax[1, 1]); ax[1, 1].set_title("Q-Q of log(amount) vs Normal (log-normal check)")
s = np.sort(obs.values)[::-1]; cs = np.cumsum(s) / s.sum()
ax[1, 2].plot(np.arange(1, len(s) + 1) / len(s) * 100, cs * 100, color="crimson"); ax[1, 2].plot([0, 100], [0, 100], "k--", lw=1)
ax[1, 2].set_xlabel("% of days (wettest first)"); ax[1, 2].set_ylabel("% of total rain"); ax[1, 2].set_title("Concentration of rainfall")
plt.tight_layout(); plt.show()

print("Gamma fit to rain-day amounts: shape = %.2f, scale = %.1f mm | KS statistic = %.3f (p = %.3g; with n this large KS flags tiny deviations)" % (a_hat, scale_hat, ks.statistic, ks.pvalue))
print("\n10 wettest days:"); display(obs.sort_values(ascending=False).head(10).to_frame("rain_mm"))
print("INSIGHT: gamma shape %.2f (<1 means many light-rain days and a long tail of heavy ones)." % a_hat)

### 💡 Insight
- The histogram confirms the **point mass at zero**; the log-axis histogram shows that *given* rain, amounts span more than three orders of magnitude.
- A **Gamma distribution** (flexible, non-negative, skewed) describes the rain-day amounts reasonably well in the Q-Q plot; its shape parameter below 1 means *many light-rain days and a long tail of downpours*. We use it in the hurdle models: **P(rain) × Gamma(amount | rain)**. Deviations in the extreme tail are expected — extremes are the hard part of rainfall modelling.
- The concentration curve shows how few days carry most of the water. The top days are **kept** — removing them would blind the model to exactly the events that cause floods.

### 3.3 Seasonality — which times of year are wet, and *why* (more rain days or heavier rain?)

In [ ]:
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
d = pd.DataFrame({"rain": obs, "wet": (obs >= WET_THR).astype(float)})
d["month"] = d.index.month
clim = d.groupby("month").apply(lambda g: pd.Series({
    "mean mm/day": g.rain.mean(), "rain-day freq %": 100 * g.wet.mean(),
    "mean on rain days (mm)": g.rain[g.wet == 1].mean(), "p95 on rain days (mm)": g.rain[g.wet == 1].quantile(.95),
    "max (mm)": g.rain.max()}))
clim.index = month_names
display(clim.round(1))

doy = np.minimum(d.index.dayofyear.values, 365)
def circ_smooth(s, w=31):
    h = w // 2
    return pd.concat([s.iloc[-h:], s, s.iloc[:h]]).rolling(w, center=True).mean().iloc[h:-h]
wf = circ_smooth(d.groupby(doy).wet.mean()); ma = circ_smooth(d.groupby(doy).rain.mean())

fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
ax[0].bar(month_names, clim["rain-day freq %"], color="steelblue"); ax[0].set_title("Rain-day frequency by month (%)")
ax[1].bar(month_names, clim["mean on rain days (mm)"], color="slateblue"); ax[1].set_title("Mean amount on rain days (mm)")
ax[2].plot(wf.index, 100 * wf.values, color="steelblue", label="rain-day freq (%)"); ax2 = ax[2].twinx()
ax2.plot(ma.index, ma.values, color="crimson", label="mean mm/day"); ax2.grid(False)
ax[2].set_xlabel("day of year"); ax[2].set_title("Smoothed (31-day) annual cycle"); ax[2].legend(loc="upper left"); ax2.legend(loc="upper right")
plt.tight_layout(); plt.show()
wettest, driest = clim["mean mm/day"].idxmax(), clim["mean mm/day"].idxmin()
print(f"INSIGHT: wettest month on average = {wettest} ({clim['mean mm/day'].max():.1f} mm/day), driest = {driest} ({clim['mean mm/day'].min():.1f} mm/day); "
      f"rain-day frequency ranges {clim['rain-day freq %'].min():.0f}%-{clim['rain-day freq %'].max():.0f}%.")

### 💡 Insight
- The annual cycle exists in **both parts of the problem**: how *often* it rains (frequency) and how *hard* it rains when it does (mean amount per rain day). Both curves are smooth, which is why a handful of sine/cosine terms (Fourier terms) can represent them with very few parameters (Stage C).
- The smoothed curves show the **monsoon peak and dry trough**; this calendar information is the cheapest and most reliable predictive signal at long lead times — we will use it as the benchmark every model has to beat (**day-of-year climatology**).
- Because frequency and intensity follow different seasonal patterns, modelling *occurrence* and *amount* with separate seasonal curves is more faithful than a single curve for the daily mean.

### 3.4 Long-term trend (annual totals and rain-day frequency)

In [ ]:
cnt = y_raw.groupby(y_raw.index.year).count()
full_years = cnt[cnt >= 360].index
ann = pd.DataFrame({"total_mm": y_raw.groupby(y_raw.index.year).sum(), "wet_days": (y_raw >= WET_THR).groupby(y_raw.index.year).sum()}).loc[full_years]
sl, ic, r, p, se = stats.linregress(ann.index, ann.total_mm)
sl2, _, r2_, p2, _ = stats.linregress(ann.index, ann.wet_days)
rho, p_s = stats.spearmanr(ann.index, ann.total_mm)

fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
ax[0].bar(ann.index, ann.total_mm, color="steelblue"); ax[0].plot(ann.index, ic + sl * ann.index, "r--", lw=2, label=f"trend {sl:+.1f} mm/yr (p={p:.2f})"); ax[0].legend(); ax[0].set_title("Annual total rainfall (complete years)")
ax[1].bar(ann.index, ann.wet_days, color="teal"); ax[1].plot(ann.index, ann.wet_days.mean() + 0 * ann.index, "k:", label="mean"); ax[1].set_title(f"Number of rain days per year (trend {sl2:+.2f} days/yr, p={p2:.2f})"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"Annual total: {sl:+.2f} mm/yr, R2={r**2:.3f}, p={p:.3f} | Spearman rho={rho:.2f}, p={p_s:.3f} | rain days: {sl2:+.2f} days/yr, p={p2:.3f}")
print("INSIGHT:", "statistically significant trend in annual rainfall." if p < 0.05 else "NO statistically significant long-term trend in annual rainfall at the 5% level.")

### 💡 Insight
- Only *complete* years are used (a partial last year would bias the annual total).
- If the p-values are large, there is **no significant trend** in either the annual total or the number of rain days → **no trend term** is needed in the daily models. If a trend is detected, add a slowly varying term (e.g. `year`) to the occurrence/amount models.
- Year-to-year swings in annual totals are large; they reflect weather regimes (e.g. monsoon strength, ENSO) that are not visible in the rainfall history alone.

### 3.5 Wet/dry spells — is today's weather informative about tomorrow's?

In [ ]:
wet_i = (y >= WET_THR).astype(int)
tab = pd.crosstab(wet_i.shift(1).iloc[1:].astype(int).rename("yesterday"), wet_i.iloc[1:].rename("today"))
p_ww, p_dw = tab.loc[1, 1] / tab.loc[1].sum(), tab.loc[0, 1] / tab.loc[0].sum()
chi2, p_chi, *_ = stats.chi2_contingency(tab)
p_wet = wet_i.mean()
print(f"P(rain)               = {p_wet:.3f}")
print(f"P(rain | rain yesterday) = {p_ww:.3f}")
print(f"P(rain | dry yesterday)  = {p_dw:.3f}   -> lift = {p_ww/p_dw:.2f}x | chi-square independence test p = {p_chi:.2g}")

# by calendar month
tm = pd.DataFrame({"y": wet_i.shift(1), "t": wet_i, "m": y.index.month}).dropna()
by_m = tm.groupby("m").apply(lambda g: pd.Series({"P(rain|rain)": g.t[g.y == 1].mean(), "P(rain|dry)": g.t[g.y == 0].mean()}))
by_m.index = month_names

# spell lengths
runs = wet_i.groupby((wet_i != wet_i.shift()).cumsum())
spell = pd.DataFrame({"state": runs.first(), "len": runs.size()})
fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
by_m.plot(kind="bar", ax=ax[0], rot=0); ax[0].set_ylim(0, 1); ax[0].set_title("Chance of rain given yesterday's state, by month")
for st_, col, lab in [(1, "steelblue", "wet spells"), (0, "darkorange", "dry spells")]:
    vc = spell[spell.state == st_]["len"].value_counts(normalize=True).sort_index().loc[:15]
    ax[1].plot(vc.index, vc.values, marker="o", color=col, label=f"{lab} (mean {spell[spell.state == st_]['len'].mean():.1f} d)")
ax[1].set_yscale("log"); ax[1].set_xlabel("spell length (days)"); ax[1].set_title("Wet and dry spell length distribution"); ax[1].legend()
exp_wet = 1 / (1 - p_ww)
ax[2].bar(["observed wet spell", "if days were independent"], [spell[spell.state == 1]["len"].mean(), 1 / (1 - p_wet)], color=["steelblue", "lightgrey"]); ax[2].set_title("Mean wet-spell length (days)")
plt.tight_layout(); plt.show()
print(f"INSIGHT: rain is {p_ww/p_dw:.2f}x more likely after a rain day than after a dry day (independent days would give 1.00x).")

### 💡 Insight
- This is the **key predictability test for daily data**. If days were independent, `P(rain | rain yesterday)` would equal `P(rain | dry yesterday)`. A clear gap (and a tiny chi-square p-value) means **weather persists for a day or two** — which is the only edge a model can get beyond the calendar at short lead times.
- The gap is usually modest, and it fades within a few days — we will see this as the forecast skill decaying from 1-day to 7-day lead time.
- Wet-spell lengths are longer than they would be for independent days; the spell distributions are geometric-like, which supports a **first-order Markov / lag-feature** occurrence model (Stage D).

### 3.6 Autocorrelation (ACF / PACF)

In [ ]:
ylog = np.log1p(y)
wet_anom = (y >= WET_THR).astype(float) - (y >= WET_THR).astype(float).groupby(y.index.month).transform("mean")   # rain/no-rain minus its monthly average

fig, ax = plt.subplots(2, 2, figsize=(16, 8))
plot_acf(ylog, lags=40, ax=ax[0, 0]); ax[0, 0].set_title("ACF — log(1+rain), 40 days")
plot_pacf(ylog, lags=40, ax=ax[0, 1], method="ywm"); ax[0, 1].set_title("PACF — log(1+rain), 40 days")
plot_acf(wet_anom, lags=40, ax=ax[1, 0]); ax[1, 0].set_title("ACF — rain/no-rain after removing the monthly mean")
plot_acf(ylog, lags=400, ax=ax[1, 1], alpha=None); ax[1, 1].set_title("ACF — long lags (annual cycle visible near lag 365)")
plt.tight_layout(); plt.show()

ac = acf(ylog, nlags=400, fft=True); ac_w = acf(wet_anom, nlags=40, fft=True)
print("ACF log-rain at lags 1,2,3,7,14,30,365:", {l: round(float(ac[l]), 3) for l in (1, 2, 3, 7, 14, 30, 365)})
print("ACF of de-seasonalised rain/no-rain at lags 1,2,3,7:", {l: round(float(ac_w[l]), 3) for l in (1, 2, 3, 7)})
lb = acorr_ljungbox(ylog, lags=[10, 30], return_df=True); display(lb.round(4))
print(f"INSIGHT: lag-1 autocorrelation = {ac[1]:.2f}; by lag 7 it is {ac[7]:.2f}. Beyond a few days the history carries almost no information.")

### 💡 Insight
- Autocorrelation is **positive but small at lag 1–2 and drops to ~0 within about a week** — the series has *short* memory. A model fed only with rainfall history therefore cannot be skilful for long lead times.
- After removing the monthly average, the rain/no-rain indicator still shows lag-1 correlation: this is the **weather persistence** the memory features exploit.
- At lag ≈ 365 a small seasonal echo remains, but it is weak compared with the noise → *'same day last year'* is a poor forecast (we test it as a baseline).
- Ljung-Box rejects 'no autocorrelation', so memory features are justified — but the magnitude says to **expect modest skill**.

### 3.7 Stationarity tests (ADF & KPSS)

In [ ]:
def stationarity(s, name):
    s = pd.Series(s).dropna()
    adf_stat, adf_p = adfuller(s, maxlag=40, autolag="AIC")[:2]
    kp_stat, kp_p = kpss(s, regression="c", nlags="auto")[:2]
    verdict = ("stationary" if (adf_p < .05 and kp_p >= .05) else "non-stationary" if (adf_p >= .05 and kp_p < .05) else "conflicting / borderline")
    return {"series": name, "ADF stat": round(adf_stat, 2), "ADF p": round(adf_p, 4), "KPSS stat": round(kp_stat, 3), "KPSS p (capped .01-.1)": round(kp_p, 3), "verdict": verdict}

tests = pd.DataFrame([stationarity(y, "daily rainfall"), stationarity(ylog, "log(1+rain)"),
                      stationarity(y.rolling(30).mean(), "30-day rolling mean")])
display(tests)
print("H0 of ADF = unit root (non-stationary);  H0 of KPSS = stationary.")

### 💡 Insight
- Daily rainfall (and its log) is **stationary**: no unit root, no drifting level. With ~14 000 points, KPSS can occasionally flag tiny deviations — read it together with the trend test in 3.4.
- **Decision: no differencing.** The thing left to model is the *calendar cycle*, which the stationarity tests do not remove (they only look for trends/unit roots) — we model it explicitly with day-of-year climatology or Fourier terms.

### 3.8 Predictability ceiling — how much can calendar and yesterday explain?

In [ ]:
dd = pd.DataFrame({"amount": obs, "wet": (obs >= WET_THR).astype(float)})
dd["month"] = dd.index.month; dd["doy"] = np.minimum(dd.index.dayofyear.values, 365)
dd["prev_wet"] = (y.shift(1) >= WET_THR).astype(float).reindex(dd.index)
dd = dd.dropna()

def r2(target, keys):
    g = dd.groupby(keys)[target].transform("mean")
    return 1 - ((dd[target] - g) ** 2).sum() / ((dd[target] - dd[target].mean()) ** 2).sum()

rows = []
for lab, keys in [("calendar month", ["month"]), ("calendar month + yesterday wet/dry", ["month", "prev_wet"])]:
    rows.append({"predictors": lab, "R2 of daily amount (%)": 100 * r2("amount", keys), "R2 of rain/no-rain (%)": 100 * r2("wet", keys)})
ceil_tab = pd.DataFrame(rows).set_index("predictors"); display(ceil_tab.round(2))
print("INSIGHT: calendar + yesterday explain only %.1f%% of the day-to-day variance of rainfall amounts -> the rest is weather noise that history cannot predict." % ceil_tab.iloc[1, 0])

### 💡 Insight
- This is the **ceiling for a history-only model**: even grouping by month *and* yesterday's wet/dry state explains only a small share of daily variance (in-sample, so it is optimistic).
- Most of what happens tomorrow is *weather*, determined by atmospheric conditions that rainfall history does not contain. Therefore: judge every model by **skill relative to climatology** (not by absolute R²), and expect gains mainly in the **probability of rain** at lead time 1–2 days.
- This also tells us what would break the ceiling: weather-forecast inputs (numerical weather prediction, radar nowcasts, satellite) or climate drivers — see *Next steps*.

### 3.9 EDA summary & modelling decisions
| Finding | Where | Modelling decision |
|---|---|---|
| Daily data zero-inflated, rain-day amounts heavy-tailed (Gamma-like) | 2.2, 3.2 | **Hurdle model:** P(rain) × Gamma(amount \| rain); no log transform of the target |
| Strong annual cycle in *both* frequency and intensity | 3.3 | Seasonal terms in both parts (Fourier terms / day-of-year climatology) |
| No significant long-term trend (check p-values) | 3.4 | No trend term |
| Weather persists for ~1–3 days (P(rain\|rain) > P(rain\|dry)) | 3.5, 3.6 | Add **lag / spell features** for short lead times |
| Memory vanishes within about a week | 3.6 | Direct forecasting per lead time; expect skill → climatology at 7+ days |
| Stationary series | 3.7 | No differencing |
| Calendar + yesterday explain little variance | 3.8 | Judge models by **skill vs climatology**; add probabilistic scores (Brier, AUC) |
| Same-day measurements of other variables unknown in advance | — | Only past rainfall + calendar are used as predictors (no leakage) |

## 4. Train / Test Split and Validation Design
Time-series data must **never be shuffled**: the test set is always the *most recent* period.
- **Test set:** last **730 days** (2 years) — touched once at the end.
- **Model selection:** **rolling-origin (expanding-window) cross-validation** on the training part: 5 folds, each fits on all earlier data and forecasts the next **365 days** (a full seasonal cycle, so every fold contains a wet and a dry season).
- **Lead times:** every model is evaluated for forecasts made **1, 3 and 7 days ahead**. For a target day *t* and lead time *h*, the model may use observations up to day *t − h* only.

In [ ]:
train, test = y.iloc[:-TEST_DAYS], y.iloc[-TEST_DAYS:]
assert len(train) - N_FOLDS * CV_STEP >= 3 * 365, "not enough history: reduce N_FOLDS / TEST_DAYS"
print(f"Train: {train.index[0].date()} -> {train.index[-1].date()}  ({len(train)} days)")
print(f"Test : {test.index[0].date()} -> {test.index[-1].date()}  ({len(test)} days)")

fold_windows = []
for k in range(N_FOLDS, 0, -1):
    end = len(train) - k * CV_STEP
    fold_windows.append((train.index[0].date(), train.index[end - 1].date(), train.index[end].date(), train.index[end + CV_H - 1].date()))
display(pd.DataFrame(fold_windows, columns=["train_start", "train_end", "val_start", "val_end"], index=[f"fold {i}" for i in range(1, N_FOLDS + 1)]))

fig, ax = plt.subplots(figsize=(15, 3.8))
ax.plot(train.rolling(30).mean(), label="train (30-day mean)", color="steelblue"); ax.plot(test.rolling(30).mean(), label="test (30-day mean)", color="crimson")
for i, (_, _, vs, ve) in enumerate(fold_windows):
    ax.axvspan(pd.Timestamp(vs), pd.Timestamp(ve), color="orange", alpha=.25, label="CV validation windows" if i == 0 else None)
ax.legend(); ax.set_ylabel("mm/day"); ax.set_title("Chronological split and CV windows"); plt.tight_layout(); plt.show()
print(f"Mean rain: train {train.mean():.2f} mm/day | test {test.mean():.2f} mm/day | rain-day frequency: train {(train >= WET_THR).mean():.1%} vs test {(test >= WET_THR).mean():.1%}")

### 💡 Insight
- Each CV fold trains only on data *before* its validation window; the five orange bands cover five different years, so the score is not decided by one unusually wet or dry year. 5 folds × 365 days ≈ 1 800 validation forecasts per model and lead time.
- **No leakage:** features for a target day are built from days ≤ *t − h*; rolling windows look backwards only; missing days were filled using the training period only.
- The last line compares the climate of the test years with the training years: if they differ noticeably (a wetter/drier stretch), every model's test **bias** will reflect it — that is a property of the period, not of one model.

## 5. Forecaster Classes (written to `rain_forecasters.py`)
All models share one interface — `fit(y, h)` and `predict(history, dates)` → DataFrame(`forecast`, `p_rain`, `q_upper`) — so they are scored by exactly the same code.

**Every model is a hurdle model**: it outputs `p_rain` (probability of ≥ 0.2 mm) and `mu` (mean amount on a rain day).
- `forecast = p_rain × mu` — the expected rainfall in mm (the RMSE-optimal point forecast);
- `q_upper` = 95 % upper bound from a Gamma distribution for the amounts.

**Features** (used by the memory models; only information up to the forecast origin *t − h*): log-rain of the last 0/1/2/3/6 days, wet/dry flags of the last 3 days, rolling mean rainfall and rain-day share over 3/7/14/30 days, days since last rain, length of the current wet spell, and Fourier terms of the **target day's** position in the year.

Writing the classes to a `.py` file also makes the saved model loadable later (pickle needs the class definitions).

In [ ]:
%%writefile rain_forecasters.py
import copy
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression, GammaRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import (HistGradientBoostingClassifier, HistGradientBoostingRegressor,
                              RandomForestClassifier, RandomForestRegressor)

warnings.filterwarnings("ignore")

WET_THR = 0.2          # a "rain day" = at least 0.2 mm (same definition NEA uses)
WARMUP = 30            # longest look-back window used by the lag features (days)


# --------------------------------------------------------------------------------------
# Feature engineering
# --------------------------------------------------------------------------------------
def fourier_terms(idx, K):
    """K pairs of sin/cos terms for the annual cycle (period = 365.25 days)."""
    doy = np.asarray(idx.dayofyear, float)
    cols = {}
    for k in range(1, K + 1):
        cols[f"sin{k}"] = np.sin(2 * np.pi * k * doy / 365.25)
        cols[f"cos{k}"] = np.cos(2 * np.pi * k * doy / 365.25)
    return pd.DataFrame(cols, index=idx)


def lag_features(y):
    """Features known at the END of each day d (the forecast 'origin'): only data up to and including d."""
    y = y.asfreq("D")
    wet = (y >= WET_THR).astype(float)
    F = pd.DataFrame(index=y.index)
    for l in (0, 1, 2, 3, 6):
        F[f"amt_l{l}"] = np.log1p(y.shift(l))
    for l in (0, 1, 2):
        F[f"wet_l{l}"] = wet.shift(l)
    for w in (3, 7, 14, 30):
        F[f"mean_{w}"] = y.rolling(w).mean()
        F[f"wetfrac_{w}"] = wet.rolling(w).mean()
    pos = np.arange(len(y))
    last_wet = pd.Series(np.where(wet.values == 1, pos, np.nan), index=y.index).ffill()
    F["days_since_wet"] = (pos - last_wet).clip(upper=30).fillna(30)
    run = wet.groupby((wet != wet.shift()).cumsum()).cumsum()          # length of current wet spell (0 if dry)
    F["wet_streak"] = run.clip(upper=30)
    return F


def design(y_hist, targets, h, K):
    """Feature matrix for forecasting each target date `h` days ahead from its origin (target - h)."""
    targets = pd.DatetimeIndex(targets)
    F = lag_features(y_hist).reindex(targets - pd.Timedelta(days=h))
    F.index = targets
    return pd.concat([F, fourier_terms(targets, K)], axis=1)


# --------------------------------------------------------------------------------------
# Base class: every model returns P(rain) and the mean amount on a wet day
# --------------------------------------------------------------------------------------
class RainModel:
    """Common interface:  fit(y, h)  and  predict(y_hist, dates) -> DataFrame(forecast, p_rain, q_upper).

    forecast = expected rainfall (mm) = P(rain) x mean amount on a wet day  (the RMSE-optimal point forecast)
    p_rain   = probability that the day gets >= 0.2 mm
    q_upper  = upper prediction bound (default 95 %), from a gamma distribution for wet-day amounts.
    `y_hist` = observed daily series up to (at least) the forecast origin of every requested date.
    """
    name = "base"
    h_ = 1
    shape_ = 0.8

    def fit(self, y, h=1):
        raise NotImplementedError

    def _pm(self, y_hist, dates):
        raise NotImplementedError

    def _fit_shape(self, y):
        """Gamma shape of wet-day amounts around the seasonal mean (same for all models -> comparable intervals)."""
        c = Climatology()._fit_core(y)
        _, mu = c._pm(None, y.index)
        w = (y.values >= WET_THR)
        phi = np.mean((y.values[w] / mu[w] - 1.0) ** 2)
        self.shape_ = float(np.clip(1.0 / phi, 0.2, 5.0))

    def predict(self, y_hist, dates, q=0.95):
        dates = pd.DatetimeIndex(dates)
        p, mu = self._pm(y_hist, dates)
        p = np.clip(np.asarray(p, float), 1e-4, 1 - 1e-4)
        mu = np.clip(np.asarray(mu, float), WET_THR, None)
        upper = np.zeros(len(p))
        need = q > (1 - p)                                   # otherwise the q-quantile is a dry day (0 mm)
        u = (q - (1 - p[need])) / p[need]
        upper[need] = stats.gamma.ppf(u, a=self.shape_, scale=mu[need] / self.shape_)
        return pd.DataFrame({"forecast": p * mu, "p_rain": p, "q_upper": upper}, index=dates)


# --------------------------------------------------------------------------------------
# Stage A/B: baselines and climatology
# --------------------------------------------------------------------------------------
class HistoricalMean(RainModel):
    def fit(self, y, h=1):
        self.h_ = h
        w = y.values >= WET_THR
        self.p_, self.mu_ = float(w.mean()), float(y.values[w].mean())
        self._fit_shape(y); return self
    def _pm(self, y_hist, dates):
        return np.full(len(dates), self.p_), np.full(len(dates), self.mu_)


class Persistence(RainModel):
    """Tomorrow (or h days ahead) = today's value."""
    lag_days = None
    def fit(self, y, h=1):
        self.h_ = h
        w = y.values >= WET_THR
        self.mu_ = float(y.values[w].mean())
        self._fit_shape(y); return self
    def _pm(self, y_hist, dates):
        lag = self.h_ if self.lag_days is None else self.lag_days
        v = y_hist.reindex(dates - pd.Timedelta(days=lag)).values
        wet = v >= WET_THR
        return wet.astype(float), np.where(wet, v, self.mu_)


class SameDayLastYear(Persistence):
    lag_days = 365


class Climatology(RainModel):
    """For every day of the year: average over all past years within +/- `window` days."""
    def __init__(self, window=15):
        self.window = window
    def _fit_core(self, y):
        doy = np.minimum(y.index.dayofyear.values, 365) - 1
        wet = (y.values >= WET_THR).astype(float)
        n = np.bincount(doy, minlength=365).astype(float)
        nw = np.bincount(doy, weights=wet, minlength=365)
        amt = np.bincount(doy, weights=np.where(wet == 1, y.values, 0.0), minlength=365)
        sm = lambda a: sum(np.roll(a, s) for s in range(-self.window, self.window + 1))
        self.p_ = sm(nw) / np.maximum(sm(n), 1)
        self.mu_ = sm(amt) / np.maximum(sm(nw), 1)
        return self
    def fit(self, y, h=1):
        self.h_ = h
        self._fit_core(y)
        self._fit_shape(y)
        return self
    def _pm(self, y_hist, dates):
        doy = np.minimum(np.asarray(dates.dayofyear), 365) - 1
        return self.p_[doy], self.mu_[doy]


# --------------------------------------------------------------------------------------
# Stage C: harmonic (Fourier) hurdle model - seasonality only, no memory
# --------------------------------------------------------------------------------------
class HarmonicHurdle(RainModel):
    """P(rain) = logistic(Fourier terms);  amount | rain = Gamma GLM (log link) on Fourier terms."""
    def __init__(self, K=3):
        self.K = K
    def fit(self, y, h=1):
        self.h_ = h
        X = fourier_terms(y.index, self.K).values
        wet = y.values >= WET_THR
        self.clf_ = LogisticRegression(C=1e3, max_iter=2000).fit(X, wet)
        self.reg_ = GammaRegressor(alpha=0.0, max_iter=2000).fit(X[wet], y.values[wet])
        self._fit_shape(y); return self
    def _pm(self, y_hist, dates):
        X = fourier_terms(dates, self.K).values
        return self.clf_.predict_proba(X)[:, 1], self.reg_.predict(X)


# --------------------------------------------------------------------------------------
# Stage D/E: hurdle models with memory (lag features): GLM, gradient boosting, random forest
# --------------------------------------------------------------------------------------
class LagHurdle(RainModel):
    def __init__(self, kind="glm", K=3, seed=42):
        self.kind, self.K, self.seed = kind, K, seed

    def _make(self):
        s = self.seed
        if self.kind == "glm":
            clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=3000))
            reg = make_pipeline(StandardScaler(), GammaRegressor(alpha=1e-2, max_iter=3000))
        elif self.kind == "hgb":
            kw = dict(max_depth=3, learning_rate=0.05, max_iter=150, min_samples_leaf=50, l2_regularization=1.0, random_state=s)
            clf, reg = HistGradientBoostingClassifier(**kw), HistGradientBoostingRegressor(loss="gamma", **kw)
        elif self.kind == "rf":
            clf = RandomForestClassifier(n_estimators=100, min_samples_leaf=40, max_features=0.5, n_jobs=-1, random_state=s)
            reg = RandomForestRegressor(n_estimators=100, min_samples_leaf=20, max_features=0.5, n_jobs=-1, random_state=s)
        else:
            raise ValueError(self.kind)
        return clf, reg

    def fit(self, y, h=1):
        self.h_ = h
        targets = y.index[WARMUP + h:]
        X = design(y, targets, h, self.K)
        t = y.loc[targets]
        ok = X.notna().all(axis=1).values
        wet = (t.values >= WET_THR)
        self.cols_ = list(X.columns)
        clf, reg = self._make()
        self.clf_ = clf.fit(X.values[ok], wet[ok])
        self.reg_ = reg.fit(X.values[ok & wet], t.values[ok & wet])
        self._fit_shape(y); return self

    def _pm(self, y_hist, dates):
        X = design(y_hist, dates, self.h_, self.K)[self.cols_].fillna(0.0).values
        return self.clf_.predict_proba(X)[:, 1], self.reg_.predict(X)


# --------------------------------------------------------------------------------------
# Deployment wrapper: one direct model per horizon (1..H days ahead)
# --------------------------------------------------------------------------------------
class RainForecastBundle:
    def __init__(self, prototype, horizons=range(1, 8)):
        self.prototype, self.horizons = prototype, list(horizons)

    def fit(self, y):
        self.models_ = {h: copy.deepcopy(self.prototype).fit(y, h) for h in self.horizons}
        self.last_date_ = y.index[-1]
        return self

    def predict_next(self, y_hist, q=0.95):
        """Forecast days +1 ... +H after the LAST observation in `y_hist` (needs >= 30 days of history)."""
        y_hist = y_hist.asfreq("D")
        last = y_hist.index[-1]
        rows = []
        for h, m in self.models_.items():
            d = pd.DatetimeIndex([last + pd.Timedelta(days=h)])
            r = m.predict(y_hist, d, q=q).iloc[0]
            rows.append({"date": d[0], "lead_days": h, "p_rain": r.p_rain, "expected_mm": r.forecast, f"upper_{int(q*100)}_mm": r.q_upper})
        return pd.DataFrame(rows).set_index("date")

In [ ]:
import importlib, rain_forecasters
importlib.reload(rain_forecasters)
from rain_forecasters import (HistoricalMean, Persistence, SameDayLastYear, Climatology, HarmonicHurdle,
                              LagHurdle, RainForecastBundle, fourier_terms)
print("Forecaster classes loaded:", ["HistoricalMean", "Persistence", "SameDayLastYear", "Climatology", "HarmonicHurdle", "LagHurdle", "RainForecastBundle"])

### 💡 Insight
- A single interface lets very different models (a constant, a calendar lookup, a GLM, boosted trees) be compared with identical code — no risk of an unfair comparison.
- Hurdle outputs make the models comparable on **two** questions: *"how many mm?"* (RMSE, MAE) and *"will it rain?"* (Brier score, AUC) — the second is usually what an operations team acts on.
- The Gamma shape for the 95 % bound is estimated once from the training rain days (same for all models), so interval coverage differences come from the models' `p_rain` and `mu`, not from different interval recipes.

## 6. Evaluation Harness (metrics, rolling CV, plots)

In [ ]:
def metrics(actual, fc, keep):
    a = np.asarray(actual, float)[keep]
    f, p, u = fc["forecast"].values[keep], fc["p_rain"].values[keep], fc["q_upper"].values[keep]
    wet_ = (a >= WET_THR).astype(int); err = a - f
    return {"MAE": np.mean(np.abs(err)), "RMSE": np.sqrt(np.mean(err ** 2)), "Bias": err.mean(),
            "Brier": np.mean((p - wet_) ** 2), "AUC": roc_auc_score(wet_, p) if 0 < wet_.sum() < len(wet_) else np.nan,
            "Cover95": np.mean(a <= u)}

RESULTS, FACTORIES = {}, {}
observed = ~MISSING.reindex(y.index).values          # True where the day was really measured

def evaluate(name, factory, horizons=HORIZONS, verbose=True):
    # Rolling-origin CV on train -> fit on full train -> 730-day forecast on test, for every lead time
    t0 = time.time(); FACTORIES[name] = factory; RESULTS[name] = {}
    for h in horizons:
        folds, oos = [], []
        for k in range(N_FOLDS, 0, -1):
            end = len(train) - k * CV_STEP
            tr, va = y.iloc[:end], y.iloc[end:end + CV_H]
            fc = factory().fit(tr, h).predict(y, va.index)           # history = observations up to the origin (target - h)
            keep = observed[end:end + CV_H]
            m = metrics(va.values, fc, keep); m["fold"] = N_FOLDS - k + 1; folds.append(m)
            oos.append(fc.assign(actual=va.values, source="cv")[keep])
        cv = pd.DataFrame(folds).set_index("fold")
        model = factory().fit(train, h)
        fc_t = model.predict(y, test.index)
        keep_t = observed[len(train):]
        mt = metrics(test.values, fc_t, keep_t)
        oos.append(fc_t.assign(actual=test.values, source="test")[keep_t])
        RESULTS[name][h] = {"cv": cv, "test": mt, "fc_test": fc_t, "model": model, "oos": pd.concat(oos)}
        if verbose:
            print(f"{name} | lead {h}d\n  CV   (5x365d): RMSE={cv.RMSE.mean():6.3f}  MAE={cv.MAE.mean():6.3f}  Brier={cv.Brier.mean():.4f}  AUC={cv.AUC.mean():.3f}"
                  f"\n  TEST (730d)  : RMSE={mt['RMSE']:6.3f}  MAE={mt['MAE']:6.3f}  Brier={mt['Brier']:.4f}  AUC={mt['AUC']:.3f}  Bias={mt['Bias']:+.2f}  cover95={mt['Cover95']:.2f}")
    print(f"  [{time.time() - t0:.1f}s]\n")

def plot_forecasts(names, h=H_MAIN, days=120, title=""):
    t = test.iloc[-days:]; pal = sns.color_palette("tab10", len(names))
    fig, ax = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
    ax[0].bar(t.index, t.values, width=1.0, color="lightgrey", label="actual (mm)")
    for n, c in zip(names, pal):
        f = RESULTS[n][h]["fc_test"].loc[t.index]; ax[0].plot(f.index, f["forecast"], color=c, lw=1.5, label=f"{n}")
    ax[0].set_ylabel("mm"); ax[0].legend(ncol=3, fontsize=8); ax[0].set_title(f"{title} — expected rainfall, lead {h} day(s), last {days} test days")
    ax[1].vlines(t.index[t.values >= WET_THR], 0, 1, color="lightgrey", lw=2, label="rain day")
    for n, c in zip(names, pal):
        f = RESULTS[n][h]["fc_test"].loc[t.index]; ax[1].plot(f.index, f["p_rain"], color=c, lw=1.5)
    ax[1].set_ylabel("P(rain)"); ax[1].set_ylim(0, 1); ax[1].legend(loc="upper right")
    plt.tight_layout(); plt.show()

### 💡 Insight
- **Point-forecast metrics:** MAE and RMSE in mm (RMSE punishes missing the heavy days). **Bias** = actual − forecast (negative = over-forecasting).
- **Probability metrics:** the **Brier score** (mean squared error of P(rain); lower is better; always-predict-the-base-rate is the reference), **AUC** (can the model rank wet days above dry days? 0.5 = no skill, 1 = perfect) and **Cover95** (share of days with actual ≤ the 95 % upper bound — should be ≈ 0.95).
- `MAPE` and MASE are deliberately **not used**: a day with 0 mm makes percentage errors undefined, and a seasonal-naive scale is meaningless for a zero-inflated series. Instead we report **skill relative to day-of-year climatology** (section 8).
- Days with filled (missing) observations are removed from every score.

## 7. Models — start from scratch, then improve step by step

### Stage A · Baselines (the bar every real model must beat)
#### M0 — Historical mean   ·   M1 — Persistence (value *h* days ago)   ·   M2 — Same day last year

In [ ]:
evaluate("M0 Historical mean", lambda: HistoricalMean())
evaluate("M1 Persistence", lambda: Persistence())
evaluate("M2 Same day last year", lambda: SameDayLastYear())
plot_forecasts(["M0 Historical mean", "M1 Persistence", "M2 Same day last year"], title="Simple baselines")

### 💡 Insight
- **M0** predicts the same expected rainfall and the long-run rain frequency every day: the *know-nothing* benchmark. Its RMSE is hard to beat by much, because daily rainfall is so noisy.
- **M1 Persistence** copies the observation from *h* days ago. For *rain/no-rain* this is a hard 0/1 "probability", so its Brier score is poor, and for amounts it doubles the noise → typically **worse than the mean**. In a series with weak memory, copying one noisy value adds noise instead of information.
- **M2 Same-day-last-year** uses a single past year as the seasonal pattern → also poor. Lesson (same as in the monthly notebook): **to use seasonality we must average over many years, not copy one**.

#### M3 — Day-of-year climatology (average of the same time of year, ±15 days, over all past years) — the real benchmark

In [ ]:
evaluate("M3 Climatology", lambda: Climatology(window=15))
plot_forecasts(["M3 Climatology"], title="Day-of-year climatology")

### 💡 Insight
- Averaging the same calendar window over all training years captures the **annual cycle in both the rain probability and the amount**. It improves on M0 in Brier score and RMSE: the yearly cycle is genuine, usable signal.
- Its forecast is the same for lead times 1, 3 and 7 (it ignores recent weather), which is why it is the natural reference: **a model is only useful if it beats climatology**. All skill scores in section 8 are measured against M3.
- The forecast curve is smooth and low; the observed rain days are spikes far above it — this is expected for an *expected-value* forecast of a zero-inflated variable.

### Stage B · Harmonic hurdle model (smooth seasonality with few parameters)
Climatology needs 365 × 2 values and is noisy day to day. A **logistic regression** for P(rain) and a **Gamma GLM** (log link) for the amount, both on `K` sine/cosine pairs of the day of year, describe the *smooth* annual cycle with only `2K+1` parameters per part. `K` is chosen by AIC on the training data.

In [ ]:
wd_tr = train[train >= WET_THR]; occ = (train >= WET_THR).astype(int)
rows = []
for K in range(1, 7):
    r1 = sm.GLM(occ, sm.add_constant(fourier_terms(train.index, K)), family=sm.families.Binomial()).fit()
    r2_ = sm.GLM(wd_tr, sm.add_constant(fourier_terms(wd_tr.index, K)), family=sm.families.Gamma(sm.families.links.Log())).fit()
    rows.append({"K": K, "AIC occurrence": r1.aic, "AIC amount": r2_.aic, "AIC total": r1.aic + r2_.aic, "BIC total": r1.bic_llf + r2_.bic_llf})
kgrid = pd.DataFrame(rows).set_index("K"); display(kgrid.round(1))
K_BEST = int(kgrid["AIC total"].idxmin()); K_BIC = int(kgrid["BIC total"].idxmin())
print(f"Chosen K = {K_BEST} harmonics by AIC (BIC would choose K = {K_BIC})")

evaluate("M4 Harmonic hurdle", lambda: HarmonicHurdle(K=K_BEST))

# fitted seasonal curves vs monthly raw frequencies
m4 = RESULTS["M4 Harmonic hurdle"][1]["model"]
idx_y = pd.date_range("2019-01-01", "2019-12-31", freq="D")
p4, mu4 = m4._pm(None, idx_y)
mon_f = (train >= WET_THR).groupby(train.index.month).mean()
fig, ax = plt.subplots(1, 2, figsize=(15, 4.3))
ax[0].plot(idx_y.dayofyear, p4, color="crimson", label=f"harmonic K={K_BEST}"); ax[0].scatter((np.arange(12) * 30.4 + 15), mon_f.values, color="k", label="monthly frequency"); ax[0].set_title("P(rain) over the year"); ax[0].legend()
ax[1].plot(idx_y.dayofyear, mu4, color="crimson"); ax[1].set_title("Mean amount on a rain day (mm) over the year")
plt.tight_layout(); plt.show()
plot_forecasts(["M3 Climatology", "M4 Harmonic hurdle"], title="Climatology vs harmonic hurdle")

### 💡 Insight
- AIC/BIC pick a **small number of harmonics** — the annual cycle is smooth, so 2–4 sine/cosine pairs capture it. The fitted curves follow the monthly points closely while being smooth day to day.
- M4 is expected to be **about as good as, or slightly better than, the climatology lookup**: same information (the calendar), fewer parameters, less noise. If the two are practically tied, prefer the simpler/smoother M4 for deployment.
- Like M3, it has **no memory of recent weather**, so it cannot improve between lead times — the next stage adds that memory.

### Stage C · Hurdle model with weather memory (logistic + Gamma GLMs on lag features)
The EDA showed that rain persists for a day or two. Here the same two-part model also receives **lag and spell features** (yesterday's rain, rain-day share of the last 3–30 days, days since last rain, ...) plus Fourier terms. Linear models keep it interpretable and stable on this noisy target.

In [ ]:
evaluate("M5 Hurdle GLM + memory", lambda: LagHurdle("glm", K=3, seed=SEED))

m5 = RESULTS["M5 Hurdle GLM + memory"][1]["model"]
coef = pd.Series(m5.clf_.steps[-1][1].coef_[0], index=m5.cols_).sort_values()
fig, ax = plt.subplots(figsize=(8, 6)); coef.plot(kind="barh", ax=ax, color=np.where(coef > 0, "steelblue", "indianred"))
ax.set_title("Standardised logistic coefficients — P(rain tomorrow), lead 1 day"); plt.tight_layout(); plt.show()
plot_forecasts(["M3 Climatology", "M5 Hurdle GLM + memory"], title="Climatology vs GLM with memory")

### 💡 Insight
- The coefficient plot (lead 1 day) shows **which recent-weather features move the rain probability**: positive bars raise it (e.g. recent rainy days, a current wet spell), negative bars lower it (e.g. long time since last rain). Large coefficients on the first few lags and short rolling windows agree with the EDA — persistence is *short-term*.
- **Expected outcome:** a clear gain in **Brier score / AUC at lead 1 day** over climatology, shrinking at 3 and 7 days (see section 8). RMSE in mm improves much less: even when we know whether it will rain, *how much* is largely unpredictable.

### Stage D · Machine learning on the same features (Gradient Boosting, Random Forest)
Trees can capture non-linear effects and interactions (for example *'rain yesterday AND wet last week AND monsoon season'*) that the GLM cannot. Both are two-part models: a classifier for occurrence and a regressor (Gamma loss for boosting) for the wet-day amount.

In [ ]:
evaluate("M6 Gradient Boosting", lambda: LagHurdle("hgb", K=3, seed=SEED))
evaluate("M7 Random Forest", lambda: LagHurdle("rf", K=3, seed=SEED))

rf_m = RESULTS["M7 Random Forest"][1]["model"]
imp = pd.Series(rf_m.clf_.feature_importances_, index=rf_m.cols_).sort_values()
imp.tail(15).plot(kind="barh", figsize=(8, 5), color="seagreen", title="Random Forest feature importance — P(rain), lead 1 day"); plt.tight_layout(); plt.show()
plot_forecasts(["M5 Hurdle GLM + memory", "M6 Gradient Boosting", "M7 Random Forest"], title="Statistical vs machine-learning forecasts")

### 💡 Insight
- Feature importances show **which memory signals matter**: short windows (last 1–3 days, current spell length, 3–7-day rain-day share) and the seasonal terms, typically not long windows. This agrees with the ACF.
- With a **weak, short-lived signal**, flexible models gain little or nothing over the well-specified GLM and can lose by fitting noise; this is the same lesson as the monthly notebook ('flexible ML does not help when the signal-to-noise ratio is this low'). Heavily regularised settings are used on purpose (shallow trees, large leaves).
- Compare CV and test scores in the next section before trusting any ML gain: with this noise level, differences of a few thousandths in Brier score may not be significant.

## 8. Compare All Models

In [ ]:
rows = []
for n, byh in RESULTS.items():
    for h, r in byh.items():
        cv, t = r["cv"], r["test"]
        rows.append({"model": n, "h": h, "CV_RMSE": cv.RMSE.mean(), "CV_RMSE_std": cv.RMSE.std(), "CV_MAE": cv.MAE.mean(), "CV_Brier": cv.Brier.mean(), "CV_AUC": cv.AUC.mean(),
                     "Test_RMSE": t["RMSE"], "Test_MAE": t["MAE"], "Test_Brier": t["Brier"], "Test_AUC": t["AUC"], "Test_Bias": t["Bias"], "Test_Cover95": t["Cover95"]})
comp = pd.DataFrame(rows)
REF = "M3 Climatology"
ref = comp[comp.model == REF].set_index("h")
for k in ("CV_RMSE", "CV_Brier", "Test_RMSE", "Test_Brier"):
    comp[f"{k}_skill%"] = (1 - comp[k] / comp.h.map(ref[k])) * 100          # positive = better than climatology
comp["CV_composite_skill%"] = (comp["CV_RMSE_skill%"] + comp["CV_Brier_skill%"]) / 2

for h in HORIZONS:
    print(f"\n=== Lead time {h} day(s) ===")
    display(comp[comp.h == h].drop(columns="h").set_index("model").sort_values("CV_Brier").round(3))

fig, ax = plt.subplots(1, 3, figsize=(19, 5))
for n, g in comp.groupby("model"):
    if n in ("M1 Persistence", "M2 Same day last year"): continue
    g = g.sort_values("h")
    ax[0].plot(g.h, g["CV_Brier_skill%"], marker="o", label=n); ax[1].plot(g.h, g["CV_RMSE_skill%"], marker="o", label=n); ax[2].plot(g.h, g["Test_Brier_skill%"], marker="o", label=n)
for a, t_ in zip(ax, ["CV Brier skill vs climatology (%)", "CV RMSE skill vs climatology (%)", "TEST Brier skill vs climatology (%)"]):
    a.axhline(0, color="k", lw=1); a.set_xticks(list(HORIZONS)); a.set_xlabel("lead time (days)"); a.set_title(t_)
ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()

h_ = H_MAIN
fold_rmse = pd.DataFrame({n: byh[h_]["cv"]["Brier"] for n, byh in RESULTS.items()}).T
plt.figure(figsize=(9, 5)); sns.heatmap(fold_rmse, annot=True, fmt=".3f", cmap="YlOrRd"); plt.title(f"Brier score per CV fold (lead {h_}d) — lower is better"); plt.xlabel("fold"); plt.ylabel("")
plt.tight_layout(); plt.show()

### 💡 Insight
- **Read the three charts left to right:** a model is useful where its line is **above 0** (better than climatology). Persistence and last-year are excluded from the charts because they are far below zero.
- The classic pattern for daily rainfall is **skill that is largest at 1 day and decays toward 0 by about 3–7 days** — beyond that, history-only models are no better than the calendar. Lines that dip slightly *below* 0 at 3–7 days mean the memory features are fitting noise there — the model has nothing left to offer beyond the calendar. If the lines stay at 0 even at 1 day, the memory features are not helping on your data and climatology is the honest answer.
- Skill is clearer in the **probability of rain (Brier)** than in **RMSE** (mm): predicting *whether* it rains is easier than predicting *how much*.
- The fold heat-map shows year-to-year variation in the score for each model; a model whose advantage disappears in some folds is not reliably better.
- The CV and the test panels should agree in sign; if a model wins CV but loses on test, treat the gain as noise.

## 9. Select the Best Model

In [ ]:
# Selection rule: highest average CROSS-VALIDATED skill over all lead times, skill = mean of (RMSE skill, Brier skill) relative to climatology.
# The hold-out test set is NOT used for selection.
score = comp.groupby("model")["CV_composite_skill%"].mean().sort_values(ascending=False)
display(score.round(2).to_frame("avg CV skill vs climatology (%)"))
best_name = score.index[0]
print("BEST MODEL (by CV skill):", best_name)
if best_name == REF:
    print("-> No model beats day-of-year climatology in cross-validation: the honest recommendation is to ship climatology.")

sub = comp[comp.model == best_name].set_index("h")
display(sub[["CV_RMSE", "CV_Brier", "CV_AUC", "Test_RMSE", "Test_Brier", "Test_AUC", "Test_Bias", "Test_Cover95",
             "CV_RMSE_skill%", "CV_Brier_skill%", "Test_RMSE_skill%", "Test_Brier_skill%"]].round(3))
d_ = (RESULTS[REF][H_MAIN]["cv"]["Brier"] - RESULTS[best_name][H_MAIN]["cv"]["Brier"])
print(f"Brier gain vs climatology per CV fold (lead {H_MAIN}d): {d_.round(4).tolist()} -> better in {(d_ > 0).sum()} of {len(d_)} folds")
plot_forecasts([REF, best_name] if best_name != REF else [REF], h=H_MAIN, days=120, title=f"Best model: {best_name}")

### 💡 Insight
- The rule is explicit and **uses only cross-validation**; the test set is a final, independent check.
- The **per-fold Brier gain** line is the honesty check from the monthly notebook: if the best model is better than climatology in all/most folds the advantage is robust; if only in some, call it 'modest'. A model can rank first on average and still be a statistical tie with a simpler one — in that case prefer the simpler model.
- If the printed message says climatology wins, that is a *valid result*: it means the available history adds no usable information beyond the season at these lead times.

### 9.1 Are the rain probabilities trustworthy? (calibration) and where are the errors?

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(19, 5))
for h, col in zip(HORIZONS, ["#1f77b4", "#ff7f0e", "#2ca02c"]):
    o = RESULTS[best_name][h]["oos"]
    q = pd.qcut(o.p_rain, 10, duplicates="drop")
    cal = o.groupby(q, observed=True).apply(lambda g: pd.Series({"pred": g.p_rain.mean(), "obs": (g.actual >= WET_THR).mean()}))
    ax[0].plot(cal.pred, cal.obs, marker="o", color=col, label=f"lead {h}d")
ax[0].plot([0, 1], [0, 1], "k--"); ax[0].set_xlabel("predicted P(rain)"); ax[0].set_ylabel("observed frequency"); ax[0].set_title("Reliability diagram (pooled CV + test)"); ax[0].legend()

o1 = RESULTS[best_name][H_MAIN]["oos"].copy(); o1["err"] = o1.actual - o1.forecast; o1["abs_err"] = o1.err.abs(); o1["wet"] = (o1.actual >= WET_THR).astype(float)
o1["brier"] = (o1.p_rain - o1.wet) ** 2; o1["m"] = o1.index.month
by_m = o1.groupby("m").agg(MAE=("abs_err", "mean"), Brier=("brier", "mean"), mean_actual=("actual", "mean"), p_mean=("p_rain", "mean"), wet_freq=("wet", "mean")); by_m.index = month_names
by_m["MAE"].plot(kind="bar", ax=ax[1], color="indianred", rot=0); ax[1].set_title(f"MAE by calendar month (mm), lead {H_MAIN}d")
ax[2].hist(o1.p_rain, bins=30, color="slateblue"); ax[2].set_title("Distribution of forecast P(rain)")
plt.tight_layout(); plt.show()
display(by_m.round(3).T)
print(f"Pooled out-of-sample (lead {H_MAIN}d): MAE={o1.abs_err.mean():.2f} mm | share of days with error > 20 mm = {(o1.abs_err > 20).mean():.1%} | 95% upper-bound coverage = {(o1.actual <= o1.q_upper).mean():.1%}")

### 💡 Insight
- **Reliability diagram:** points near the diagonal mean the stated probabilities are honest (when the model says 70 %, it rains on ≈ 70 % of those days). Points below the diagonal = over-confident rain probabilities, above = under-confident. This matters more than RMSE for decisions based on a rain probability threshold.
- **Spread of probabilities:** a narrow histogram (all forecasts near the base rate) is the visible face of the weak signal — probabilities that never leave, say, 30–60 % are an *honest* reflection of what history can predict.
- **By calendar month:** errors in mm are largest in the wettest months (more rain, more variance); the Brier score shows in which seasons the occurrence forecast is least reliable.
- **Interval coverage:** the 95 % upper bound should cover about 95 % of days; clearly lower coverage means extreme days are still under-estimated — see *Next steps*.

## 10. Final Model: Re-fit on ALL Data and Forecast the Next 7 Days
In deployment we need one model **per lead time**, so the selected model type is re-fitted on all data for lead times 1…7 (`RainForecastBundle`).

In [ ]:
FUTURE_H = 7
bundle = RainForecastBundle(FACTORIES[best_name](), horizons=range(1, FUTURE_H + 1)).fit(y)
future = bundle.predict_next(y)
display(future.round(3))

fig, ax = plt.subplots(1, 2, figsize=(15, 4.3))
ax[0].bar(future.index.strftime("%a %d %b"), 100 * future.p_rain, color="steelblue"); ax[0].set_ylim(0, 100); ax[0].axhline(100 * (y >= WET_THR).mean(), color="k", ls=":", label="long-run rain-day frequency"); ax[0].legend()
ax[0].set_ylabel("%"); ax[0].set_title("P(rain) — next 7 days")
ax[1].bar(future.index.strftime("%a %d %b"), future.expected_mm, color="teal", label="expected (mm)"); ax[1].plot(future.index.strftime("%a %d %b"), future.upper_95_mm, "r_", ms=18, mew=3, label="95% upper bound")
ax[1].set_ylabel("mm"); ax[1].set_title("Expected rainfall and 95% upper bound — next 7 days"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"Forecast issued after the last observation ({y.index[-1].date()}). Expected total over the next {FUTURE_H} days: {future.expected_mm.sum():.1f} mm.")

### 💡 Insight
- The table is the operational product: for each coming day a **rain probability**, an **expected amount** and a **95 % upper bound** for 'how bad could it be'.
- Expect probabilities to be closest to the climatological base rate for the later days (the forecast converges to climatology as memory fades) and to deviate most on **day +1**.
- The expected amount is a mean over rain and no-rain outcomes (so it is typically small, even on days with a high rain probability); use the **upper bound** for flood/drainage thresholds and the **probability** for go/no-go decisions on outdoor work.
- If the data ends in the past (e.g. June 2020), this forecast window is already history: compare it with the observed days as an extra out-of-time check.

## 11. Save the Best Model

In [ ]:
SAVE_DIR = pathlib.Path("saved_model"); SAVE_DIR.mkdir(exist_ok=True)

artifact = {"bundle": bundle, "name": best_name, "trained_until": str(y.index[-1].date()), "unit": "mm/day",
            "target": "daily rainfall total (Changi)", "synthetic_data": USING_SYNTHETIC, "horizons": list(range(1, FUTURE_H + 1))}
joblib.dump(artifact, SAVE_DIR / "best_daily_rainfall_forecaster.joblib")

meta = {
    "best_model": best_name,
    "selection_rule": "highest average cross-validated skill vs day-of-year climatology (mean of RMSE skill and Brier skill) over lead times 1/3/7 days; 5 expanding-window folds x 365 days",
    "data": {"station": "Changi Climate Station", "from": str(y.index[0].date()), "to": str(y.index[-1].date()), "n_days": int(len(y)), "missing_days": int(MISSING.sum()), "synthetic": USING_SYNTHETIC},
    "wet_day_threshold_mm": WET_THR,
    "validation_by_lead_time": comp[comp.model == best_name].set_index("h")[["CV_RMSE", "CV_Brier", "CV_AUC", "Test_RMSE", "Test_Brier", "Test_AUC", "CV_Brier_skill%", "CV_RMSE_skill%"]].round(4).to_dict(orient="index"),
    "usage": "from rain_forecasters import *; import joblib; b = joblib.load(path)['bundle']; b.predict_next(recent_daily_rainfall_series)",
}
json.dump(meta, open(SAVE_DIR / "model_card.json", "w"), indent=2, default=str)
comp.round(4).to_csv(SAVE_DIR / "model_comparison.csv", index=False)
future.round(3).to_csv(SAVE_DIR / "forecast_next_7_days.csv")
for p_ in sorted(SAVE_DIR.iterdir()):
    print(f"{p_}  ({p_.stat().st_size / 1e3:.1f} KB)")

### 💡 Insight
- The `.joblib` holds the **7 fitted models** (one per lead time), the selected model's name and the training end-date. `model_card.json` records the selection rule, data period, and validation scores (including a `synthetic` flag); `model_comparison.csv` is the audit trail; the forecast CSV holds the next 7 days.
- To load the artifact elsewhere keep **`rain_forecasters.py`** next to it (pickle stores classes by reference).

## 12. Reload the Model & Forecast (deployment check)

In [ ]:
def forecast_rainfall(history, path=SAVE_DIR / "best_daily_rainfall_forecaster.joblib"):
    # history: pandas Series of recent DAILY rainfall (mm) indexed by date; the last date is 'today' (needs >= 30 days).
    a = joblib.load(path)
    out = a["bundle"].predict_next(history).round(3)
    out.index = out.index.strftime("%Y-%m-%d")
    return out

loaded = joblib.load(SAVE_DIR / "best_daily_rainfall_forecaster.joblib")
print("Loaded:", loaded["name"], "| trained until", loaded["trained_until"], "| synthetic data:", loaded["synthetic_data"])
reloaded_fc = forecast_rainfall(y)
same = np.allclose(reloaded_fc["p_rain"].values, future["p_rain"].round(3).values) and np.allclose(reloaded_fc["expected_mm"].values, future["expected_mm"].round(3).values)
print("Reloaded model reproduces the in-memory forecast:", same)
display(reloaded_fc)

# API demo: issue a forecast 'as of' 7 days before the end of the data and compare with what actually happened
# (illustration of the interface only - this model has already seen these days in training, so it is NOT a validation)
asof = y.iloc[:-FUTURE_H]
demo = forecast_rainfall(asof).assign(actual_mm=y.iloc[-FUTURE_H:].values)
display(demo)

### 💡 Insight
- The reloaded model reproduces the in-memory forecast (**`True`**), so the saved artifact is complete; `forecast_rainfall(history)` can be wrapped in an API or a scheduled daily job that passes the **latest 30+ days of observations** and returns the next 7 days.
- The *as-of* demo only shows the interface (inputs: recent daily rainfall; output: next-7-day table). It is **not** evidence of accuracy, because those days were part of the final fit — the evidence is in sections 8–9 (cross-validation and hold-out test).
- For production: refit as new data arrives (fitting takes seconds), monitor Brier score and calibration **against climatology**, and fall back to climatology if the advantage disappears.

## 13. Conclusion & Next Steps

In [ ]:
print("SUMMARY (all numbers computed from this run)\n")
print("Data:", "SYNTHETIC (demo only)" if USING_SYNTHETIC else f"{y.index[0].date()} -> {y.index[-1].date()}, {len(y)} days, {int(MISSING.sum())} missing")
print("Best model:", best_name, "\n")
tbl = comp[comp.model == best_name].set_index("h")[["CV_Brier_skill%", "CV_RMSE_skill%", "Test_Brier_skill%", "Test_RMSE_skill%", "Test_AUC", "Test_Cover95"]].round(2)
tbl.index = [f"lead {h} d" for h in tbl.index]; display(tbl)

**What we did**
1. **Re-framed the problem for daily data:** the target is zero-inflated and heavy-tailed, so every model is a **hurdle model** — *P(rain)* × *Gamma amount* — instead of a (log-)Gaussian monthly model.
2. **Data audit & EDA** on the daily series: zero-inflation, Gamma-like wet-day amounts, an annual cycle in both frequency and intensity, no significant trend, short weather memory (wet/dry spells), stationarity, and a low predictability ceiling.
3. **Leak-free validation:** chronological split (last 2 years as test) + 5-fold rolling-origin CV with 365-day windows, at lead times of 1, 3 and 7 days.
4. **Seven models, each motivated by the weakness of the previous**: baselines → day-of-year climatology → harmonic hurdle → hurdle with weather memory → gradient boosting / random forest.
5. **Selected on cross-validation only** (skill vs climatology), checked on the hold-out test, calibration-checked, re-fitted on all data as 7 direct models (lead 1…7), saved and reloaded.

**Key lessons**
- Daily rainfall is **mostly weather noise**: a history-only model has real, but **modest and short-lived skill** (rain probability at 1–2 days), and converges to climatology within about a week. This is the honest ceiling, not a modelling failure.
- Always compare against **day-of-year climatology**, not just against the mean or persistence: it is the benchmark that exposes whether a model adds anything.
- Evaluate **probabilities** (Brier, AUC, reliability), not only mm-errors: for zero-inflated rainfall, *'will it rain?'* is both more predictable and more actionable than *'how many mm?'*.

**Possible improvements**
- Add **forecast inputs** known in advance: numerical weather prediction output, radar/satellite nowcasts, humidity/pressure/temperature/wind from the same station (as *lagged* predictors), monsoon-phase and ENSO/IOD indices — the single most promising way past the noise ceiling.
- Predict **intensity classes / quantiles** (e.g. P(> 20 mm), P(> 50 mm)) directly with quantile or classification models for flood-planning decisions; model the extreme tail with an extreme-value (GPD) component.
- Probabilistic ensembles (average of climatology + GLM + boosting) and season-specific calibration (isotonic) of the rain probabilities.
- Multi-station / spatial models, and sub-daily (hourly) data for nowcasting.
- Operations: daily retraining, monitoring of Brier skill vs climatology, and an API around `forecast_rainfall()`.